## 1、TextSplitter的使用

1、使用细节 \
1:TextSplitter作为各种具体的文档拆分器的父类 \
2:内部定义了一些常用的属性： \
chunk_size:返回块的最大尺寸，单位是字符数。默认值为4000（由长度函数测量）\
chunk_overlap:相邻两个块之间的字符重叠数，避免信息在边界处被切断而丢失。默认值为200，通常会设置为chunk_size的10%-20% \
length_function:用于测量给定字符块中保留分隔数，默认值为False \
keep_separator:是否在块中保留分隔符，默认值为False \
add_start_index:如果为True，则在元数据中包含块的起始索引。默认值为False \
strip_whitespace:如果为True，则从每个文档的开始和结束处去除空白字符。默认值为True \

内部定义的常用方法：\
情况1:按照字符串进行拆分：\
split_text(xxx)：传入的参数类型：<font color='red'>字符串</font>；返回值的类型：list[str] \
create_documents(xxx):传入的参数类型：<font color='red'>list[str]</font>;返回值的类型：list[Document]。底层调用了split_text(xxx)

情况2:按照Document对象进行拆分 \
split_documents(xxx):传入的参数类型：<font color='red'>list[Document]</font>;返回值的类型：list[Document]。底册调用create_documents(xxx)

2、Document对象和Str是什么关系？ \
文档拆分器可以按照字符串进行切分，也可以按照Document进行切分。其中，Str可以理解为是Document对象的page_content属性

### 具体的拆分器1:CharacterTextSplitter。Split by character

In [8]:
# 举例1:体会chunk_size、chunk_overlap
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

# 字符串示例 一个空格也是一个字符数
text = """
LangChain是一个用于开发语言模型驱动的应用程序的框架。它提供块一套工具和抽象方法，使开发者，
能够更容易地构建复杂的应用程序
"""
textSplitter=CharacterTextSplitter(
    # 若必须禁用分隔符，则实际块长略小于chunk_size（相当于默认了空格）
    separator='',  # 设置为空字符串时，表示禁用分隔符优先
    chunk_size=50, # 每块大小
    chunk_overlap=5, # 块与块之间的重复字符数
)
# 分隔文本
texts=textSplitter.split_text(text)
# print(texts)

for i, chunk in enumerate(texts):
    print(f'块{i+1}:长度{len(chunk)}')
    print(chunk)
    print('-'*50)

块1:长度49
LangChain是一个用于开发语言模型驱动的应用程序的框架。它提供块一套工具和抽象方法，使开发者
--------------------------------------------------
块2:长度22
，使开发者，
能够更容易地构建复杂的应用程序
--------------------------------------------------


In [45]:
# 举例2：体会separator
from langchain_text_splitters import CharacterTextSplitter

# 字符串示例 一个空格也是一个字符数
text = """
这是一个示例文本啊。我们将使用CharacterTextSplitter将其分割成小块。分隔给予字符数。
"""
"""流程：
    1、先按separator分隔
    2、发现前2个块小于chunk_size的值就合并，如果大于，第一个块就不会并，接着看第二块和第三块
"""

textSplitter=CharacterTextSplitter(
    # 若必须禁用分隔符，则实际块长略小于chunk_size（相当于默认了空格）
    separator='。',  # 按句号分隔（分隔符优先）
    # 使用30的时候报错Created a chunk of size 33, which is longer than the specified 30
    # chunk_size=30, # 每块大小

    # 会把2个块合并成一个
    # chunk_size=44, # 每块大小

    # 会把后面2个块合并
    chunk_size=42, # 每块大小

    chunk_overlap=0, # 块与块之间的重复字符数
)
# 分隔文本
texts=textSplitter.split_text(text)
# print(texts)

for i, chunk in enumerate(texts):
    print(f'块{i+1}:长度{len(chunk)}')
    print(chunk)
    print('-'*50)

块1:长度9
这是一个示例文本啊
--------------------------------------------------
块2:长度41
我们将使用CharacterTextSplitter将其分割成小块。分隔给予字符数
--------------------------------------------------


In [57]:
# 举例3:体会chunk_overlap
from langchain_text_splitters import CharacterTextSplitter

'''流程：
    1、先按separator分隔
    2、因为第一块+第二块小于chunk_size，则把1和2合并
    3、再看第二块是否小于chunk_overlap，不小于就不会有重叠部分，小于则把2和3合并
'''
# 2.自定义要分隔的文本
text='这是第一段文本。这是第二段文本。最后一段结束'

textSplitter=CharacterTextSplitter(
    separator='。',
    chunk_size=20,
    chunk_overlap=5,
    keep_separator=True # chunk中是否保留切割符
)
# 分隔文本
texts=textSplitter.split_text(text)
# print(texts)

for i, chunk in enumerate(texts):
    print(f'块{i+1}:长度{len(chunk)}')
    print(chunk)
    print('-'*50)

块1:长度15
这是第一段文本。这是第二段文本
--------------------------------------------------
块2:长度7
。最后一段结束
--------------------------------------------------


## RecursiveCharacterTextSplitter:最常用

In [64]:
# 举例1:使用split_text()方法演示
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
text = """
LangChain是一个用于开发语言模型驱动的应用程序的框架\n\n它提供块一套工具和抽象方法\n使开发者\n
能够更容易地构建复杂的应用程序
"""
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=10,
    chunk_overlap=0,
    # add_start_index=True, #元数据中包含 chunk 的起始索引
)
paragraphs=text_splitter.split_text(text)

for paragraph in paragraphs:
    print(paragraph)
    # print('----')

LangChain
是一个用于开发语言模
型驱动的应用程序的框
架
它提供块一套工具和
抽象方法
使开发者
能够更容易地构建复
杂的应用程序


In [73]:
# 举例2:使用元数据
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
text = """
LangChain是一个用于开发语言模型驱动的应用程序的框架\n\n它提供块一套工具和抽象方法\n使开发者\n
能够更容易地构建复杂的应用程序
"""
text_list=[text]
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=10,
    chunk_overlap=0,
    add_start_index=True, #元数据中包含 chunk 的起始索引 page_content='LangChain' metadata={'start_index': 1}
    # add_start_index=False, #page_content='LangChain'
)
paragraphs=text_splitter.create_documents(text_list)

for paragraph in paragraphs:
    print(paragraph)

page_content='LangChain' metadata={'start_index': 1}
page_content='是一个用于开发语言模' metadata={'start_index': 10}
page_content='型驱动的应用程序的框' metadata={'start_index': 20}
page_content='架' metadata={'start_index': 30}
page_content='它提供块一套工具和' metadata={'start_index': 33}
page_content='抽象方法' metadata={'start_index': 42}
page_content='使开发者' metadata={'start_index': 47}
page_content='能够更容易地构建复' metadata={'start_index': 53}
page_content='杂的应用程序' metadata={'start_index': 62}
